# Commented notebook: `print_peetre_decomposition` and `apply()` with NUFFT

This notebook demonstrates the Peetre decomposition pipeline from `psiop.py`,
covering **three joint-residual backends**:

1. `joint_backend='direct'` — raw symbolic joint residual.
2. `joint_backend='nufft'` — **NEW**: NUFFT phase decomposition (O(N log N)),
   with automatic fallback to low-rank then direct.
3. `joint_backend='lowrank'` — Chebyshev/SVD separable factorization.

It also shows how to **numerically apply** the operator via `apply()` / `apply_peetre()`
with each backend, and compares accuracy and timing.

## What `print_peetre_decomposition` does

The method calls `peetre_decomposition` and pretty-prints the result.
Its most important options are:

| Option | Description |
|---|---|
| `joint_backend='direct'` | Print the raw joint residual terms. |
| `joint_backend='nufft'` | Try NUFFT phase decomposition; on failure fall back to lowrank → direct. |
| `joint_backend='lowrank'` | Approximate the joint residual by separable pairs via Chebyshev/SVD. |
| `joint_bounds` | Required for `'lowrank'` and `'nufft'` printing (no grid available). |
| `joint_degree` | Chebyshev degree per variable. |
| `joint_tol` | SVD truncation tolerance. |
| `separable_local` | Forwarded to `peetre_decomposition`. |

### NUFFT cascade

When `joint_backend='nufft'`, the fallback chain is:

```
NUFFT decomposition succeeds?
  ├─ YES → apply_nufft / apply_nufft_2d   (O(N log N))
  └─ NO  → try low-rank Chebyshev/SVD
              ├─ success & error OK → separable pairs
              └─ failure or error too large → direct KN quadrature
```

NUFFT applies to joint symbols of the form
`c(x)·g(ξ)·exp(iλ(x)μ(ξ))`, e.g. `sin(x·ξ)`, `exp(i·x·ξ)`, chirps.
Rational or Gaussian joint kernels like `1/(1+(x−ξ)²)` are **not**
NUFFT-decomposable and will cascade to the next tier.

## Imports

In [1]:
import time
import numpy as np

import sympy as sp
from psiop import PseudoDifferentialOperator


## 1D example: local, separable, and joint terms

- `xi**2` is **local** (polynomial in frequency).
- `x*sin(xi)` is **separable**: `a(x) * q(xi)`.
- `1/(1+(x-xi)**2)` is **genuinely joint** (not NUFFT-compatible).

In [2]:
x, xi = sp.symbols('x xi', real=True)

# 1D symbol with three Peetre classes.
# The joint term 1/(1+(x-xi)^2) is NOT NUFFT-decomposable.
p1 = xi**2 + x * sp.sin(xi) + 1 / (1 + (x - xi)**2)

op1 = PseudoDifferentialOperator(p1, [x], mode='symbol')


## Default printing: `joint_backend='direct'`

In [3]:
print('=' * 70)
print('DEFAULT: joint_backend=direct, separable_local=False')
print('=' * 70)
op1.print_peetre_decomposition()


DEFAULT: joint_backend=direct, separable_local=False
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


## Option: `separable_local=True`

Local polynomial terms are exposed as separable pairs `a(x)*q(xi)`.

In [4]:
print('separable_local=True')
op1.print_peetre_decomposition(separable_local=True)


separable_local=True
--- 0 local term(s), polynomial in (xi,) ---
--- 2 separable non-local term(s) ---
  (1) * (xi**2)
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = 0
separable_symbol = x*sin(xi) + xi**2
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


## Option: `joint_backend='lowrank'`

The joint residual is approximated on a bounded rectangle by
`p_joint(x, xi) ≈ Σ_k a_k(x) q_k(xi)`.
`joint_bounds` is required because no numerical grid is available here.

In [5]:
print('joint_backend=lowrank')
op1.print_peetre_decomposition(
    joint_backend='lowrank',
    joint_bounds={x: (-5, 5), xi: (-30, 30)},
    joint_degree=6,
)


joint_backend=lowrank
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual factorized into 5 low-rank term(s) via factorize_symbolic (rel_l2_error=1.432e+00) ---
  (0.00026926*x**6 - 0.013732*x**4 + 0.21358*x**2 - 1.0019) * (7.2629e-9*xi**6 - 1.3078e-5*xi**4 + 0.0069955*xi**2 - 0.99805)
  (1.0742e-5*x**6 - 0.00054136*x**4 + 0.0091446*x**2 + 0.011449) * (9.7625e-10*xi**6 - 1.5949e-6*xi**4 + 0.00065437*xi**2 + 0.0093696)
  (6.8068e-7*x**5 + 7.4843e-5*x**3 + 0.0091005*x) * (1.1576e-8*xi**5 - 1.9081e-5*xi**3 + 0.0080346*xi)
  (5.5743e-6*x**6 - 0.00029603*x**4 + 0.0036664*x**2 - 0.0026623) * (-1.6396e-10*xi**6 + 2.3675e-7*xi**4 - 7.4572e-5*xi**2 + 0.0047266)
  (-1.3656e-6*x**5 - 0.00010283*x**3 + 0.0025124*x) * (-1.3111e-9*xi**5 + 1.7515e-6*xi**3 - 0.00039665*xi)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


## NEW: `joint_backend='nufft'` — oscillatory joint symbols

The NUFFT backend targets joint residuals of the form

```
p_joint(x, ξ) = Σ_k c_k(x) · g_k(ξ) · exp(i λ_k(x) μ_k(ξ))
```

Typical examples: `sin(x·ξ)`, `exp(i·x·ξ)`, chirps `exp(i·x²·ξ)`.

**This symbol is NUFFT-friendly**: the joint part `sin(x*xi)` decomposes
into two exponential terms via Euler's formula.

In [6]:
x, xi = sp.symbols('x xi', real=True)

# Joint part sin(x*xi) IS NUFFT-decomposable:
#   sin(x*xi) = (exp(i*x*xi) - exp(-i*x*xi)) / (2i)
# Each exponential is c(x)*g(xi)*exp(i*lambda(x)*mu(xi))
# with lambda(x)=x, mu(xi)=xi.
p_nufft = xi**2 + x * sp.sin(xi) + sp.sin(x * xi)

op_nufft = PseudoDifferentialOperator(p_nufft, [x], mode='symbol')

print('=' * 70)
print('NUFFT-friendly symbol: joint part = sin(x*xi)')
print('=' * 70)
op_nufft.print_peetre_decomposition(joint_backend='nufft')


NUFFT-friendly symbol: joint part = sin(x*xi)
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  sin(x*xi)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = sin(x*xi)


## NUFFT fallback: non-oscillatory joint symbol

The rational kernel `1/(1+(x−ξ)²)` is **not** of the form
`c(x)·g(ξ)·exp(iλ(x)μ(ξ))`, so the NUFFT decomposition returns `None`
and the cascade falls back to **lowrank → direct**.

A warning is emitted: *"NUFFT decomposition not applicable"*.

In [7]:
# This joint part is NOT NUFFT-decomposable.
# The nufft backend will warn and fall back to lowrank, then direct.
print('=' * 70)
print('NUFFT fallback: joint part = 1/(1+(x-xi)^2)')
print('=' * 70)
op1.print_peetre_decomposition(joint_backend='nufft')


NUFFT fallback: joint part = 1/(1+(x-xi)^2)
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


## NUFFT with a chirp and separable amplitude

`x**2 * exp(i*x*xi) * cos(xi)` has:
- spatial amplitude `c(x) = x**2`,
- spectral factor `g(ξ) = cos(ξ)`,
- phase `λ(x)·μ(ξ) = x·ξ`.

This is a textbook NUFFT-compatible term.

In [8]:
x, xi = sp.symbols('x xi', real=True)

p_chirp = xi**2 + x**2 * sp.exp(sp.I * x * xi) * sp.cos(xi)
op_chirp = PseudoDifferentialOperator(p_chirp, [x], mode='symbol')

print('Chirp symbol: joint part = x^2 * exp(i*x*xi) * cos(xi)')
op_chirp.print_peetre_decomposition(joint_backend='nufft')


Chirp symbol: joint part = x^2 * exp(i*x*xi) * cos(xi)
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 0 separable non-local term(s) ---
--- 1 irreducible joint term(s) ---
  x**2*exp(I*x*xi)*cos(xi)
local_symbol = xi**2
separable_symbol = 0
joint_symbol = x**2*exp(I*x*xi)*cos(xi)


## Smoother joint kernel in 1D (low-rank comparison)

A Gaussian bump `exp(−(x−ξ)²/8)` is smooth and well suited to
low-rank Chebyshev/SVD approximation, but **not** NUFFT-decomposable.

In [9]:
x, xi = sp.symbols('x xi', real=True)

p1b = xi**2 + x * sp.sin(xi) + sp.exp(-((x - xi)**2) / 8)
op1b = PseudoDifferentialOperator(p1b, [x], mode='symbol')


## Comparing bounds and degrees for the low-rank approximation

In [10]:
for bounds, deg in [
    ({x: (-5, 5), xi: (-15, 15)}, 8),
    ({x: (-4, 4), xi: (-12, 12)}, 10),
]:
    print('bounds =', bounds, ', degree =', deg)
    op1b.print_peetre_decomposition(
        joint_backend='lowrank',
        joint_bounds=bounds,
        joint_degree=deg,
    )
    print()


bounds = {x: (-5, 5), xi: (-15, 15)} , degree = 8
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual factorized into 5 low-rank term(s) via factorize_symbolic (rel_l2_error=4.352e-01) ---
  (-3.4432e-7*x**7 + 0.00017519*x**5 - 0.0078706*x**3 - 0.026926*x) * (5.4597e-8*xi**7 - 2.9021e-5*xi**5 + 0.0049665*xi**3 - 0.27015*xi)
  (-2.0444e-6*x**8 + 0.00017141*x**6 - 0.0059783*x**4 + 0.11081*x**2 - 0.8798) * (-8.2942e-9*xi**8 + 4.6243e-6*xi**6 - 0.00087012*xi**4 + 0.0611*xi**2 - 1.1139)
  (9.7261e-7*x**8 - 8.6251e-5*x**6 + 0.0021714*x**4 + 0.0032102*x**2 + 0.15204) * (-8.3651e-9*xi**8 + 4.4135e-6*xi**6 - 0.00074419*xi**4 + 0.038716*xi**2 + 0.12941)
  (2.266e-6*x**7 + 1.4903e-5*x**5 - 0.0011314*x**3 - 0.0048597*x) * (8.219e-9*xi**7 - 3.7058e-6*xi**5 + 0.00047073*xi**3 - 0.011925*xi)
  (-1.6054e-7*x**8 - 9.6843e-6*x**6 + 6.9195e-5*x**4 + 0.0054866*x**2 - 0.015388) * (-9.3695e-10*xi**8 + 4.3579e-7*xi**6 -

---
## Numerical application via `apply()` in 1D

We now apply the operator **numerically** to a test function using
`apply()` (which dispatches to `apply_peetre()` when `backend='peetre'`).

The three joint backends are compared:

| Backend | Strategy | Cost |
|---|---|---|
| `'direct'` | Full KN quadrature on the joint symbol | O(N²) |
| `'nufft'` | NUFFT phase decomposition (if applicable) | O(N log N) |
| `'lowrank'` | Chebyshev/SVD separable pairs | O(r · N log N) |

We use `joint_backend='direct'` as the **reference** and measure the
error of the other two backends against it.

In [11]:
# --- Grid setup ---
N = 128
L = 5.0
x_grid = np.linspace(-L, L, N, endpoint=False)
dx = x_grid[1] - x_grid[0]
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)

# --- Test function: a Gaussian bump ---
u = np.exp(-x_grid**2 / 2).astype(complex)

# --- NUFFT-friendly symbol: joint part = sin(x*xi) ---
x, xi = sp.symbols('x xi', real=True)
p_apply = xi**2 + x * sp.sin(xi) + sp.sin(x * xi)
op_apply = PseudoDifferentialOperator(p_apply, [x], mode='symbol')

print(f'Grid: N={N}, L={L}, dx={dx:.4f}')
print(f'Symbol: {p_apply}')


Grid: N=128, L=5.0, dx=0.0781
Symbol: x*sin(xi) + xi**2 + sin(x*xi)


In [12]:
# --- Reference: direct joint application ---
t0 = time.time()
res_direct = op_apply.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
t_direct = time.time() - t0

# --- NUFFT backend ---
t0 = time.time()
res_nufft = op_apply.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='nufft',
    freq_window=None, clamp=np.inf,
)
t_nufft = time.time() - t0

# --- Low-rank backend (bounds inferred from grid) ---
t0 = time.time()
res_lowrank = op_apply.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='lowrank',
    joint_degree=8,
    freq_window=None, clamp=np.inf,
)
t_lowrank = time.time() - t0

# --- Error vs. direct reference ---
err_nufft = np.max(np.abs(res_nufft - res_direct))
err_lowrank = np.max(np.abs(res_lowrank - res_direct))

print(f'direct   : time = {t_direct:.4f}s')
print(f'nufft    : time = {t_nufft:.4f}s,  max |err| vs direct = {err_nufft:.3e}')
print(f'lowrank  : time = {t_lowrank:.4f}s,  max |err| vs direct = {err_lowrank:.3e}')


direct   : time = 0.0181s
nufft    : time = 0.1174s,  max |err| vs direct = 6.006e-13
lowrank  : time = 0.4124s,  max |err| vs direct = 5.595e-01


### `apply()` with a non-NUFFT joint symbol

The rational kernel `1/(1+(x−ξ)²)` cannot be decomposed by NUFFT.
When `joint_backend='nufft'` is requested, the cascade automatically
falls back to low-rank, then to direct if needed.
A warning is emitted at each fallback step.

In [13]:
# Joint part is NOT NUFFT-decomposable -> cascade: nufft -> lowrank -> direct
p_fallback = xi**2 + x * sp.sin(xi) + 1 / (1 + (x - xi)**2)
op_fallback = PseudoDifferentialOperator(p_fallback, [x], mode='symbol')

print('Requesting nufft backend on a non-NUFFT symbol (expect warnings):')
res_fb = op_fallback.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='nufft',
    joint_degree=6,
    freq_window=None, clamp=np.inf,
)

# Compare against pure direct
res_fb_direct = op_fallback.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
print(f'max |fallback - direct| = {np.max(np.abs(res_fb - res_fb_direct)):.3e}')


Requesting nufft backend on a non-NUFFT symbol (expect warnings):
max |fallback - direct| = 2.695e-01


/home/philippe/psipy/src/psiop.py:2718: UserWarning: NUFFT decomposition not applicable; falling back to low-rank then direct.
  warnings.warn(


---
## 2D example

The symbol contains:
- local: `xi**2 + eta**2`,
- separable: `x*y*cos(xi + eta)`,
- joint Gaussian: `exp(−((x−ξ)² + (y−η)²)/8)`.

In [14]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

p2 = (
    xi**2
    + eta**2
    + x * y * sp.cos(xi + eta)
    + sp.exp(-((x - xi)**2 + (y - eta)**2) / 8)
)

op2 = PseudoDifferentialOperator(p2, [x, y], mode='symbol')


## Default 2D printing

In [15]:
print('DEFAULT 2D: joint_backend=direct')
op2.print_peetre_decomposition()


DEFAULT 2D: joint_backend=direct
--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- 1 irreducible joint term(s) ---
  exp(-eta**2/8)*exp(-x**2/8)*exp(-xi**2/8)*exp(-y**2/8)*exp(eta*y/4)*exp(x*xi/4)
local_symbol = eta**2 + xi**2
separable_symbol = x*y*cos(eta + xi)
joint_symbol = exp(-eta**2/8)*exp(-x**2/8)*exp(-xi**2/8)*exp(-y**2/8)*exp(eta*y/4)*exp(x*xi/4)


## 2D NUFFT: `sin((x+y)·ξ)` — genuinely joint phase

`sin((x+y)*xi)` has `Λ(x,y) = x+y` and `M(ξ,η) = ξ`, which embeds
into **3D** (within finufft's `nufft3d3` capability) → NUFFT passes.

By contrast, `exp(i·x·ξ) + exp(i·y·η)` has two **independent**
axis couplings that would need a 4D embedding → NUFFT rejects it
and falls back.

In [16]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# NUFFT-friendly 2D joint part: Lambda(x,y) = x+y, M(xi,eta) = xi
p2_nufft = (
    xi**2 + eta**2
    + x * y * sp.cos(xi + eta)
    + sp.sin((x + y) * xi)
)
op2_nufft = PseudoDifferentialOperator(p2_nufft, [x, y], mode='symbol')

print('2D NUFFT-friendly: joint part = sin((x+y)*xi)')
op2_nufft.print_peetre_decomposition(joint_backend='nufft')


2D NUFFT-friendly: joint part = sin((x+y)*xi)
--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- 1 irreducible joint term(s) ---
  sin(x*xi + xi*y)
local_symbol = eta**2 + xi**2
separable_symbol = x*y*cos(eta + xi)
joint_symbol = sin(x*xi + xi*y)


In [17]:
# Two independent axis couplings -> needs 4D -> NUFFT rejects, falls back
p2_reject = (
    xi**2 + eta**2
    + sp.exp(sp.I * x * xi) + sp.exp(sp.I * y * eta)
)
op2_reject = PseudoDifferentialOperator(p2_reject, [x, y], mode='symbol')

print('2D NUFFT rejection: joint = exp(i*x*xi) + exp(i*y*eta)')
print('(Expect fallback warning)')
op2_reject.print_peetre_decomposition(joint_backend='nufft')


2D NUFFT rejection: joint = exp(i*x*xi) + exp(i*y*eta)
(Expect fallback warning)
--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 0 separable non-local term(s) ---
--- 1 irreducible joint term(s) ---
  exp(I*eta*y) + exp(I*x*xi)
local_symbol = eta**2 + xi**2
separable_symbol = 0
joint_symbol = exp(I*eta*y) + exp(I*x*xi)


## 2D low-rank joint printing and timing

In [18]:
for deg in [2, 3, 4]:
    t0 = time.time()
    op2.print_peetre_decomposition(
        joint_backend='lowrank',
        joint_bounds={x: (-3, 3), y: (-3, 3), xi: (-3, 3), eta: (-3, 3)},
        joint_degree=deg,
        joint_num_samples=3000,
    )
    print('degree', deg, 'time', time.time() - t0)
    print()


--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- joint residual factorized into 9 low-rank term(s) via factorize_symbolic (rel_l2_error=2.859e-01) ---
  (-0.0029295*x**2*y**2 + 0.052254*x**2 + 0.052254*y**2 - 0.93207) * (-0.0029295*eta**2*xi**2 + 0.052254*eta**2 + 0.052254*xi**2 - 0.93207)
  (-0.011264*x**2*y + 0.0058369*x*y**2 - 0.10411*x + 0.20091*y) * (0.0058369*eta**2*xi - 0.011264*eta*xi**2 + 0.20091*eta - 0.10411*xi)
  (0.0058369*x**2*y + 0.011264*x*y**2 - 0.20091*x - 0.10411*y) * (0.011264*eta**2*xi + 0.0058369*eta*xi**2 - 0.10411*eta - 0.20091*xi)
  (-0.054938*x*y) * (-0.054938*eta*xi)
  (-4.9427e-5*x**2*y**2 - 0.049184*x**2 + 0.050213*y**2 - 0.0026288) * (-4.9427e-5*eta**2*xi**2 + 0.050213*eta**2 - 0.049184*xi**2 - 0.0026288)
  (0.006691*x**2*y**2 - 0.070016*x**2 - 0.069282*y**2 + 0.35586) * (0.006691*eta**2*xi**2 - 0.069282*eta**2 - 0.070016*xi**2 + 0.35586)
  (0.0085603*x**2*y

---
## Numerical application via `apply()` in 2D

We apply the 2D operator to a Gaussian test field and compare the
three joint backends. For the NUFFT-friendly variant we use
`sin((x+y)·ξ)` as the joint part.

In [19]:
# --- 2D grid setup ---
N2 = 32
L2 = 3.0
x_grid2 = np.linspace(-L2, L2, N2, endpoint=False)
y_grid2 = np.linspace(-L2, L2, N2, endpoint=False)
dx2 = x_grid2[1] - x_grid2[0]
dy2 = y_grid2[1] - y_grid2[0]
kx2 = 2.0 * np.pi * np.fft.fftfreq(N2, d=dx2)
ky2 = 2.0 * np.pi * np.fft.fftfreq(N2, d=dy2)

# --- Test function: 2D Gaussian ---
X2, Y2 = np.meshgrid(x_grid2, y_grid2, indexing='ij')
u2 = np.exp(-(X2**2 + Y2**2) / 2).astype(complex)

# --- NUFFT-friendly 2D symbol ---
x, y, xi, eta = sp.symbols('x y xi eta', real=True)
p2_apply = xi**2 + eta**2 + x * y * sp.cos(xi + eta) + sp.sin((x + y) * xi)
op2_apply = PseudoDifferentialOperator(p2_apply, [x, y], mode='symbol')

print(f'2D grid: N={N2}, L={L2}')


2D grid: N=32, L=3.0


In [20]:
# --- Reference: direct ---
t0 = time.time()
res2_direct = op2_apply.apply(
    u2, x_grid2, kx2,
    y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
t2_direct = time.time() - t0

# --- NUFFT ---
t0 = time.time()
res2_nufft = op2_apply.apply(
    u2, x_grid2, kx2,
    y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic',
    joint_backend='nufft',
    freq_window=None, clamp=np.inf,
)
t2_nufft = time.time() - t0

# --- Low-rank ---
t0 = time.time()
res2_lowrank = op2_apply.apply(
    u2, x_grid2, kx2,
    y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic',
    joint_backend='lowrank',
#    joint_degree=3,
#    joint_num_samples=2000,
#    freq_window=None, clamp=np.inf,
)
t2_lowrank = time.time() - t0

# --- Errors ---
err2_nufft = np.max(np.abs(res2_nufft - res2_direct))
err2_lowrank = np.max(np.abs(res2_lowrank - res2_direct))

print(f'direct   : time = {t2_direct:.4f}s')
print(f'nufft    : time = {t2_nufft:.4f}s,  max |err| = {err2_nufft:.3e}')
print(f'lowrank  : time = {t2_lowrank:.4f}s,  max |err| = {err2_lowrank:.3e}')


direct   : time = 0.1278s
nufft    : time = 0.3806s,  max |err| = 6.275e-13
lowrank  : time = 2.1408s,  max |err| = 5.358e-01


## Bonus: `apply()` with Weyl quantization

When the operator is built with `quantization='weyl'`, `apply_peetre()`
automatically converts the Weyl symbol to its Kohn–Nirenberg equivalent
via `weyl_to_kn_symbol()` before decomposing.

Example: the Weyl symbol `x*xi` becomes the KN symbol `x*xi − i/2`.

In [21]:
x, xi = sp.symbols('x xi', real=True)

# Weyl-quantized operator
p_weyl = x * xi + sp.sin(xi)
op_weyl = PseudoDifferentialOperator(
    p_weyl, [x], mode='symbol', quantization='weyl'
)

# Show the KN correction
kn_symbol = op_weyl.weyl_to_kn_symbol(order=4)
print(f'Weyl symbol:  {p_weyl}')
print(f'KN equivalent: {kn_symbol}')

# Apply with Peetre backend (Weyl -> KN conversion is automatic)
res_weyl = op_weyl.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
print(f'apply() result shape: {res_weyl.shape}')
print(f'apply() result norm:  {np.linalg.norm(res_weyl):.6e}')


Weyl symbol:  x*xi + sin(xi)
KN equivalent: x*xi + sin(xi) - I/2
apply() result shape: (128,)
apply() result norm:  4.302824e+00


## Parameter summary

| Parameter | Typical value | Meaning |
|---|---:|---|
| `joint_backend` | `'direct'` | Raw joint residual (exact, expensive). |
| `joint_backend` | `'nufft'` | **NEW**: NUFFT phase decomposition → lowrank → direct cascade. |
| `joint_backend` | `'lowrank'` | Chebyshev/SVD separable factorization. |
| `joint_bounds` | `{x: (-5,5), xi: (-30,30)}` | Bounded domain (required for printing; inferred for `apply()`). |
| `joint_degree` | `6` | Chebyshev degree per variable. |
| `joint_tol` | `1e-5` | SVD truncation tolerance. |
| `joint_num_samples` | `10000` | Monte Carlo samples for error diagnostics. |
| `joint_seed` | `42` | Random seed. |
| `joint_max_rel_error` | `None` or float | Max acceptable low-rank error before direct fallback. |
| `separable_local` | `False` / `True` | Controls local-term representation. |
| `apply_joint` | `True` | If `False`, the joint residual is skipped entirely. |
| `freq_window` | `'gaussian'` / `None` | Frequency windowing (`None` for exact math). |
| `clamp` | `1e6` / `np.inf` | Symbol magnitude clipping (`np.inf` to disable). |
| `use_cache` | `True` | Cache Peetre decomposition and low-rank pairs. |

### NUFFT-compatible joint symbols (1D)

| Symbol | NUFFT? | Reason |
|---|---|---|
| `sin(x·ξ)` | ✅ | Two exp terms via Euler. |
| `exp(i·x·ξ)` | ✅ | λ(x)=x, μ(ξ)=ξ. |
| `x²·exp(i·x·ξ)·cos(ξ)` | ✅ | Separable amplitude + phase. |
| `exp(i·x²·ξ)` | ✅ | λ(x)=x², μ(ξ)=ξ. |
| `1/(1+(x−ξ)²)` | ❌ | Rational, not exp-form. |
| `exp(−(x−ξ)²/8)` | ❌ | Gaussian, coupled envelope. |

### NUFFT-compatible joint symbols (2D)

| Symbol | NUFFT? | Reason |
|---|---|---|
| `sin((x+y)·ξ)` | ✅ | Λ=x+y, M=ξ → 3D embed. |
| `exp(i·x·ξ)` | ✅ | y trivially decoupled. |
| `exp(i·x·ξ)+exp(i·y·η)` | ❌ | Needs 4D embed (> finufft cap). |